 TICKET: DATA-147                                              
║  Título: Ingestar datos de campaña de verano de Marketing      
╠══════════════════════════════════════════════════════════════  
║  Descripción:                                                  
║  María nos ha pasado los resultados de la campaña de           
║  verano 2024 en CSVs. Necesitamos:                             
║  1. Subir los archivos a la zona raw del lake (S3)             
║  2. Validar que los datos están completos                      
║  3. Dejar una tabla limpia en la zona silver                   
║                                                                
║  Prioridad: Alta                                               
║  Asignado a: TÚ                                                
║  Fecha límite: Miércoles    

Tres archivos adjuntos en un email: `campana_verano_clientes.csv` (15.247 filas), `campana_verano_ventas.csv` (42.891 filas), y `productos_promo_verano.csv` (312 filas). Los descargas a tu máquina. Es hora de ensuciarse las manos.

**Consejo de senior:** NUNCA abras un CSV de producción directamente en Excel para "echarle un ojo". Excel modifica datos silenciosamente — convierte códigos postales a números (pierde los ceros), reformatea fechas según tu locale, y trunca filas si hay más de 1 millón. Usa SIEMPRE Pandas o la terminal para la primera exploración.

In [1]:
# explorar_campana.py — Tu primer script en FreshMart
import pandas as pd

# Cargar los 3 archivos de María
clientes = pd.read_csv('../data/campana_verano_clientes.csv')
ventas = pd.read_csv('../data/campana_verano_ventas.csv')
productos = pd.read_csv('../data/productos_promo_verano.csv')

In [2]:
print("=" * 60)
print("CLIENTES DE LA CAMPAÑA")
print("=" * 60)
clientes.info()
print()
clientes.head()

CLIENTES DE LA CAMPAÑA
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15247 entries, 0 to 15246
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   cliente_id             15247 non-null  int64 
 1   campana                15247 non-null  object
 2   fecha_envio            15247 non-null  object
 3   canal_contacto         15247 non-null  object
 4   abrio_email            9892 non-null   object
 5   clickeo_link           4182 non-null   object
 6   compro                 15247 non-null  bool  
 7   primer_compra_campana  2917 non-null   object
dtypes: bool(1), int64(1), object(6)
memory usage: 848.8+ KB



,cliente_id,campana,fecha_envio,canal_contacto,abrio_email,clickeo_link,compro,primer_compra_campana
0,104098,verano_2024,2024-06-22,email,True,False,False,NaN
1,108728,verano_2024,2024-06-22,email,True,False,False,NaN
2,113748,verano_2024,2024-06-22,email,False,NaN,False,NaN
3,108519,verano_2024,2024-06-29,sms,NaN,NaN,False,NaN
4,101770,verano_2024,2024-06-29,sms,NaN,NaN,False,NaN


In [4]:
print("-"*10,"Cantidad de nulos por columna", "-"*10)
print(clientes.isnull().sum())
print()
print(f"Clientes unicos: {clientes['cliente_id'].nunique()}")
print(f"Total de registros: {len(clientes)}")
print(f"Posibles duplicados: {len(clientes) - clientes['cliente_id'].nunique():,}")

---------- Cantidad de nulos por columna ----------
cliente_id                   0
campana                      0
fecha_envio                  0
canal_contacto               0
abrio_email               5355
clickeo_link             11065
compro                       0
primer_compra_campana    12330
dtype: int64

Clientes unicos: 14946
Total de registros: 15247
Posibles duplicados: 301


**Exploración inicial del DataFrame de clientes**

El DataFrame de clientes contiene **15.247 registros**, correspondientes a clientes que recibieron una campaña a través de diferentes canales de contacto.

Durante la exploración inicial, se identificaron **301 clientes repetidos**, lo que puede deberse a que algunos clientes recibieron la campaña a través de diferentes medios de comunicación.

Las columnas `abrio_email` y `clickeo_link` contienen una cantidad considerable de valores nulos (`Null`). Sin embargo, estos valores pueden ser esperables, ya que podrían corresponder a clientes que no abrieron el correo electrónico o no hicieron clic en el enlace.

De igual manera, la columna `primer_compra_campana` presenta varios valores nulos, lo que puede ser coherente con los clientes que no realizaron su primera compra durante la campaña.

### Explorando la data de ventas

In [5]:
print("="*60)
print("VENTAS DE LA CAMPAÑA")
print("="*60)
ventas.info()
print()
ventas.head()

VENTAS DE LA CAMPAÑA
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42891 entries, 0 to 42890
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   venta_id       42891 non-null  int64  
 1   cliente_id     42044 non-null  float64
 2   producto_id    42891 non-null  int64  
 3   fecha_compra   42891 non-null  object 
 4   cantidad       42891 non-null  int64  
 5   precio         42891 non-null  object 
 6   descuento_pct  42891 non-null  float64
 7   tienda_id      42891 non-null  int64  
 8   campana        42891 non-null  object 
dtypes: float64(2), int64(4), object(3)
memory usage: 2.9+ MB



,venta_id,cliente_id,producto_id,fecha_compra,cantidad,precio,descuento_pct,tienda_id,campana
0,8017696,105049.0,2229,14/07/2024,1,"9,89€",20.0,2,verano_2024
1,8041225,101591.0,2312,18/07/2024,1,"0,8€",0.0,2,verano_2024
2,8034956,110687.0,2167,2024-08-05,1,"4,27€",0.0,2,verano_2024
3,8043455,102181.0,3327,2024-09-12,2,"13,72€",10.0,4,verano_2024
4,8019607,113737.0,2039,2024-08-04,3,"23,25€",10.0,3,verano_2024


In [8]:
print("-"*10, "Cantidad de nulos por columna", "-"*10)
print(ventas.isnull().sum())
print()
print(f"Ventas unicas: {ventas['venta_id'].nunique()}")
print(f"Total de registros: {len(ventas)}")
print(f"Posibles duplicados: {len(ventas) - ventas['venta_id'].nunique():,}")

---------- Cantidad de nulos por columna ----------
venta_id           0
cliente_id       847
producto_id        0
fecha_compra       0
cantidad           0
precio             0
descuento_pct      0
tienda_id          0
campana            0
dtype: int64

Ventas unicas: 42736
Total de registros: 42891
Posibles duplicados: 155


**Exploración inicial del DataFrame de ventas**

A partir de la exploración inicial del DataFrame de ventas, se puede observar que el conjunto de datos contiene un total de **847 registros nulos** de cliente_id, compras de usuarios anónimos. 

Durante la revisión de los datos, se identificaron inconsistencias en el formato de algunas columnas, entre ellas:

* **`fecha_compra`**: Es de tipo `String` y contiene fechas en diferentes formatos, como `"2024-07-15"`, `"15/07/2024"` y `"Jul 15, 2024"`. Esta inconsistencia dificulta el tratamiento y análisis temporal de los datos, por lo que será necesario estandarizar el formato de las fechas.

* **`precio`**: Es de tipo `String` y contiene el símbolo del euro (`€`). Además, presenta diferentes formatos de representación decimal, como `"2,45€"` y `"0,89€"`. Es necesario revisar la consistencia de estos valores y convertir la columna a un tipo de dato numérico para facilitar los cálculos y análisis posteriores.
